## IMPORTS

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import holidays
from pytz import timezone
import regex as re
from datetime import datetime, timedelta
from dash import Dash, html, dcc
import plotly.express as px
import plotly.subplots as sp
import plotly.figure_factory as ff
import plotly.graph_objects as go
import plotly.colors as pc

/Users/ziadghanem/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


## Functions

In [2]:
eg_holidays = holidays.EG(language='en_US')

def count_items_in_order(lineitem_Qty):
    return sum([int(x) for x in lineitem_Qty])

def get_holiday_name(date):
    return eg_holidays.get(date)

def get_holiday_dates(year):
    holiday_dates = {}
    
    holiday_dates['Christmas'] = datetime(year, 12, 25)
    holiday_dates['Valentine\'s Day'] = datetime(year, 2, 14)
    holiday_dates['Mother\'s Day'] = datetime(year, 3, 21)

    holiday_dates['Black Friday'] = get_nth_weekday_of_month(year, 11, 3, 5)

    
    return holiday_dates
def is_ramadan(date_series):
    ramadan_dates = {
        2022: {'start': '2022-04-02', 'end': '2022-05-01'},
        2023: {'start': '2023-03-23', 'end': '2023-04-21'},
        2024: {'start': '2024-03-11', 'end': '2024-04-09'},
        2025: {'start': '2025-03-01', 'end': '2025-03-30'}
    }
    
    result = pd.Series(0, index=date_series.index)
    
    for year, dates in ramadan_dates.items():
        start = pd.to_datetime(dates['start'],utc=True).tz_convert('Africa/Cairo')
        end = pd.to_datetime(dates['end'],utc=True).tz_convert('Africa/Cairo')
        mask = (date_series >= start) & (date_series <= end)
        result[mask] = 1
        
    return result

def is_eid(date_series):
    eid_dates = {
        2022: {
            'eid_al_fitr_start': '2022-05-02', 'eid_al_fitr_end': '2022-05-04',
            'eid_al_adha_start': '2022-07-09', 'eid_al_adha_end': '2022-07-12'
        },
        2023: {
            'eid_al_fitr_start': '2023-04-22', 'eid_al_fitr_end': '2023-04-24',
            'eid_al_adha_start': '2023-06-28', 'eid_al_adha_end': '2023-07-01'
        },
        2024: {
            'eid_al_fitr_start': '2024-04-10', 'eid_al_fitr_end': '2024-04-12',
            'eid_al_adha_start': '2024-06-16', 'eid_al_adha_end': '2024-06-19'
        },
        2025: {
            'eid_al_fitr_start': '2025-03-31', 'eid_al_fitr_end': '2025-04-02',
            'eid_al_adha_start': '2025-06-06', 'eid_al_adha_end': '2025-06-09'
        }
    }
    
    result = pd.Series(0, index=date_series.index)
    
    for year, dates in eid_dates.items():
        fitr_start = pd.to_datetime(dates['eid_al_fitr_start'])
        fitr_end = pd.to_datetime(dates['eid_al_fitr_end'])
        fitr_mask = (date_series >= fitr_start) & (date_series <= fitr_end)
        
        adha_start = pd.to_datetime(dates['eid_al_adha_start'])
        adha_end = pd.to_datetime(dates['eid_al_adha_end'])
        adha_mask = (date_series >= adha_start) & (date_series <= adha_end)
        
        combined_mask = fitr_mask | adha_mask
        result[combined_mask] = 1
        
    return result


def get_nth_weekday_of_month(year, month, weekday, n):
    date = datetime(year, month, 1)
    
    days_until_first = (weekday - date.weekday()) % 7
    first_occurrence = date + timedelta(days=days_until_first)
    
    result = first_occurrence + timedelta(days=7 * (n - 1))
    
    return result

def days_to_next_holiday(date, holidays_dict):
    if not isinstance(date, datetime):
        date = pd.to_datetime(date)
    
    # Find the next holiday
    future_holidays = {name: holiday_date for name, holiday_date in holidays_dict.items() if holiday_date >= date}
    
    # If no future holidays in current year, look at next year
    if not future_holidays:
        next_year_holidays = get_holiday_dates(date.year + 1)
        next_holiday_name = min(next_year_holidays, key=lambda x: next_year_holidays[x])
        next_holiday_date = next_year_holidays[next_holiday_name]
    else:
        next_holiday_name = min(future_holidays, key=lambda x: future_holidays[x])
        next_holiday_date = future_holidays[next_holiday_name]
    
    # Calculate days difference
    days_difference = (next_holiday_date - date).days
    
    return days_difference, next_holiday_name

def calculate_days_to_next_shopping_holiday(dates):
    """Calculate days to next shopping holiday for a series of dates."""
    # Pre-compute holiday dates for relevant years
    years_range = range(pd.to_datetime(min(dates)).year, pd.to_datetime(max(dates)).year + 2)
    holiday_dates_by_year = {year: get_holiday_dates(year) for year in years_range}
    
    days_list, next_holiday_list = [], []
    
    for date in dates:
        date_dt = pd.to_datetime(date).tz_localize(None) if hasattr(pd.to_datetime(date), 'tz') else pd.to_datetime(date)
        year = date_dt.year
        
        future_holidays = {name: date for name, date in holiday_dates_by_year[year].items() if date >= date_dt}
        
        if not future_holidays:
            # Look at next year if no future holidays in current year
            next_year = holiday_dates_by_year[year + 1]
            next_holiday = min(next_year.items(), key=lambda x: x[1])
        else:
            next_holiday = min(future_holidays.items(), key=lambda x: x[1])
        
        days_list.append((next_holiday[1] - date_dt).days)
        next_holiday_list.append(next_holiday[0])
    
    return pd.Series(days_list), pd.Series(next_holiday_list)



In [3]:
eg_holidays

holidays.country_holidays('EG')

## FEATURE ENGENREERING

In [6]:
df = pd.read_csv('/Users/ziadghanem/Desktop/Sales Forcasting/DEPI_Graduation_project/MC_EDA/depi_grouped.csv')

In [8]:

df['Created at'] = pd.to_datetime(df['Created at'],format='%Y-%m-%d %H:%M:%S %z',utc=True).dt.tz_convert(timezone('Etc/GMT-2'))


df['Cancelled at'] = pd.to_datetime(df['Cancelled at'],format='%Y-%m-%d %H:%M:%S %z',utc=True).dt.tz_convert(timezone('Etc/GMT-2'))

df['hour'] = df['Created at'].dt.hour
df['month'] = df['Created at'].dt.month

df['day_of_week'] = df['Created at'].dt.day_name()

df['year'] = df['Created at'].dt.year
df['year_month'] = df['Created at'].dt.to_period('M')


df['season'] = pd.cut(df['Created at'].dt.month, 
                     bins=[0,3,6,9,12], 
                     labels=['Winter', 'Spring', 'Summer', 'Fall'])


df['holiday_name'] = df['Created at'].apply(get_holiday_name)
df['is_holiday'] = df['holiday_name'].notna()
df['is_weekend'] = df['Created at'].dt.dayofweek.isin([4, 5])
df['day_type'] = 'Weekday'
df.loc[df['is_weekend'], 'day_type'] = 'Weekend'
df.loc[df['is_holiday'], 'day_type'] = 'Holiday'


df['Lineitem price'] = df['Lineitem price'].str.split(',').apply(lambda x: [float(p) for p in x])
df['Lineitem name'] = df['Lineitem name'].str.split(',').apply(lambda x: [item.strip() for item in x])
df['Lineitem quantity'] = df['Lineitem quantity'].str.split(',').apply(lambda x: [int(q) for q in x])
df['Lineitem sku'] = df['Lineitem sku'].str.split(',').apply(lambda x: [sku.strip() for sku in x])
df['Lineitem sku_made'] = df['Lineitem sku_made'].str.split(',').apply(lambda x: [int(sku) for sku in x])
df['Lineitem compare at price'] = df['Lineitem compare at price'].str.split(',').apply(lambda x: [float(p) for p in x])
df['Lineitem discount'] = df['Lineitem discount'].str.split(',').apply(lambda x: [float(d) for d in x])

df['item_count']= df['Lineitem quantity'].apply(count_items_in_order)

# df.to_csv('depi_grouped.csv',index=False)

/var/folders/rd/rm1ghq0950j6l712s6ykcc5w0000gn/T/ipykernel_85387/3175242403.py:12: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  df['year_month'] = df['Created at'].dt.to_period('M')


In [9]:
df_ts = df.copy()
df_ts = df_ts.set_index('Created at')

# 1. ROLLING AVERAGES AND STATISTICS
daily_sales = df_ts.resample('D')[['Total', 'Subtotal', 'item_count']].sum()

# Create rolling features
daily_sales['total_7d_avg'] = daily_sales['Total'].rolling(window=7).mean()
daily_sales['total_30d_avg'] = daily_sales['Total'].rolling(window=30).mean()
daily_sales['items_7d_avg'] = daily_sales['item_count'].rolling(window=7).mean()
daily_sales['total_7d_std'] = daily_sales['Total'].rolling(window=7).std()

# 2. LAG FEATURES
daily_sales['total_prev_day'] = daily_sales['Total'].shift(1)
daily_sales['total_prev_week'] = daily_sales['Total'].shift(7)
daily_sales['total_prev_month'] = daily_sales['Total'].shift(30)

# Same day last week and last month
daily_sales['total_same_day_last_week'] = daily_sales['Total'].shift(7)
daily_sales['items_same_day_last_week'] = daily_sales['item_count'].shift(7)

# 3. RATIOS AND GROWTH METRICS
daily_sales['total_growth_1d'] = daily_sales['Total'] / daily_sales['total_prev_day'] - 1
daily_sales['total_growth_1w'] = daily_sales['Total'] / daily_sales['total_same_day_last_week'] - 1

# 4. SEASONALITY FEATURES
daily_sales['quarter'] = daily_sales.index.quarter
daily_sales['day_of_year'] = daily_sales.index.dayofyear
daily_sales['week_of_year'] = daily_sales.index.isocalendar().week

# 5. FOURIER TRANSFORMS FOR CYCLICAL PATTERNS
daily_sales['week_sin'] = np.sin(2 * np.pi * daily_sales.index.dayofweek / 7)
daily_sales['week_cos'] = np.cos(2 * np.pi * daily_sales.index.dayofweek / 7)

# Yearly cyclical pattern
daily_sales['year_sin'] = np.sin(2 * np.pi * daily_sales.index.dayofyear / 365)
daily_sales['year_cos'] = np.cos(2 * np.pi * daily_sales.index.dayofyear / 365)


#daily_sales['avg_item_price'] = df_ts.groupby(pd.Grouper(freq='D'))[].mean()

# Discount metrics
daily_sales['total_discount'] = df_ts.groupby(pd.Grouper(freq='D'))['Discount Amount'].sum()
daily_sales['discount_ratio'] = daily_sales['total_discount'] / daily_sales['Subtotal']

# 7. SHIPPING METRICS
# Average shipping cost per order
daily_shipping = df_ts.groupby(pd.Grouper(freq='D'))[['Shipping']].sum()
daily_order_count = df_ts.groupby(pd.Grouper(freq='D')).size()
daily_sales['avg_shipping_cost'] = daily_shipping['Shipping'] / daily_order_count


# 9. ADVANCED TIME-WINDOW FEATURES
daily_sales['unique_customers'] = df_ts.groupby(pd.Grouper(freq='D'))['Name'].nunique()
daily_sales['avg_items_per_order'] = daily_sales['item_count'] / daily_order_count

# 10. COMBINE WITH EXTERNAL FACTORS
shopping_holidays = ['Black Friday', 'Christmas', 'Valentine\'s Day', 'Mother\'s Day']


days_series, holiday_series = calculate_days_to_next_shopping_holiday(daily_sales.index)
daily_sales['days_to_next_shopping_holiday'] = days_series
daily_sales['next_shopping_holiday'] = holiday_series



daily_sales = daily_sales.reset_index()

# daily_sales['is_ramadan'] = is_ramadan(df['Created at'])
# daily_sales['is_eid'] = is_eid(daily_sales.index)

# daily_sales['is_eid_al_fitr'] = is_eid_al_fitr(daily_sales.index)  
# daily_sales['is_eid_al_adha'] = is_eid_al_adha(daily_sales.index)  
# daily_sales['pre_ramadan'] = pre_ramadan_period(daily_sales.index)  



days_to_holiday, next_holiday_name = calculate_days_to_next_shopping_holiday(daily_sales['Created at'])

daily_sales['days_to_next_shopping_holiday'] = days_to_holiday
daily_sales['next_shopping_holiday'] = next_holiday_name


daily_sales['pre_holiday_period'] = daily_sales['days_to_next_shopping_holiday'].apply(
    lambda x: 1 if x <= 7 else 0  # 1 week before holiday
)


# post_holiday_mask = daily_sales['next_shopping_holiday'].shift(-1) != daily_sales['next_shopping_holiday']
# daily_sales['post_holiday_period'] = (post_holiday_mask & (daily_sales['days_to_next_shopping_holiday'].shift(-1) > 350)).astype(int)

In [10]:
daily_sales.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 838 entries, 0 to 837
Data columns (total 30 columns):
 #   Column                         Non-Null Count  Dtype                    
---  ------                         --------------  -----                    
 0   Created at                     838 non-null    datetime64[ns, Etc/GMT-2]
 1   Total                          838 non-null    float64                  
 2   Subtotal                       838 non-null    float64                  
 3   item_count                     838 non-null    int64                    
 4   total_7d_avg                   832 non-null    float64                  
 5   total_30d_avg                  809 non-null    float64                  
 6   items_7d_avg                   832 non-null    float64                  
 7   total_7d_std                   832 non-null    float64                  
 8   total_prev_day                 837 non-null    float64                  
 9   total_prev_week                8

In [11]:
# daily_sales.to_csv('daily_sales.csv',index=False)